Mapeamento Regional e Temporal: Identificar quais regiões e estados do Brasil apresentam as maiores taxas de crimes contra mulheres e a população LGBTQIA+, bem como analisar se há uma tendência de aumento ou redução desses índices ao longo dos últimos anos.


Avaliação de Políticas Públicas: Avaliar, através de dados, se a implementação de leis de proteção específicas resultou em impactos reais na diminuição das taxas de criminalidade nas regiões analisadas.

Análise de Tipificação: Investigar a qualidade dos dados oficiais, buscando indícios de falhas de tipificação (como registrar crimes de ódio como crimes comuns).

Implicação Prática: Fornecer um panorama analítico que possa auxiliar gestores de segurança e organizações da sociedade civil na alocação de recursos (como delegacias especializadas) e no aprimoramento dos sistemas de registro de boletins de ocorrência.

# Datas:

**Início da criminalização de crimes contra LGBTQ**:

Desde junho de 2019, o STF, na ADO nº 26, criminalizou a LGBTfobia no Brasil, enquadrando atos de ódio e discriminação contra orientação sexual ou identidade de gênero na Lei de Racismo (Lei 7.716/1989). A conduta é crime inafiançável e imprescritível, com penas de 1 a 3 anos de prisão, podendo ser maior em casos de injúria racial. 
Homicídio Qualificado/Hediondo: A discriminação que motiva um homicídio (LGBTIcídio) torna o crime hediondo, com pena maior (12 a 30 anos).

**Lei do Feminicídio**:

A Lei do Feminicídio (Lei nº 13.104/2015), atualizada pela Lei nº 14.994/2024, torna o assassinato de mulheres por razões de gênero um crime autônomo e hediondo, com penas severas de 20 a 40 anos de reclusão. Considera-se feminicídio quando envolve violência doméstica/familiar ou menosprezo/discriminação à condição de mulher, com agravantes para crimes cometidos na gravidez, contra menores de 14 ou maiores de 60 anos, ou na presença de familiares.

# Perguntas

Quais regiões e estados do Brasil concentram as maiores taxas de violência contra mulheres e pessoas LGBTQIA+, e existe padrão de sazonalidade ou escalonada ao longo dos anos?

Há uma correlação observável entre a implementação de marcos legais de proteção (como a Lei do Feminicídio ou a criminalização da homofobia pelo STF) e a diminuição das ocorrências em estados específicos?

A proporção de feminicídios em relação ao total de homicídios de mulheres indica uma subnotificação ou falha na tipificação do crime em determinadas bases de dados estaduais?

Como a ausência de campos específicos para orientação sexual e identidade de gênero nos registros oficiais impacta a dimensão dos dados quando comparados aos levantamentos de ONGs e observatórios independentes?

In [17]:
from pathlib import Path
from dotenv import dotenv_values

import requests

import pandas as pd
import plotly.express as px

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

from sqlalchemy import (
    BigInteger,
    Boolean,
    Column,
    DateTime,
    Float,
    MetaData,
    String,
    Table,
    Text,
    create_engine,
    inspect,
)
from sqlalchemy.engine import URL
from sqlalchemy.schema import CreateSchema
from sqlalchemy import Integer, MetaData, Table, func, select


In [18]:
def build_postgres_engine(host: str, port: int, database: str, user: str, password: str):
    return create_engine(
        URL.create(
            "postgresql+psycopg2",
            username=user,
            password=password,
            host=host,
            port=port,
            database=database,
        )
    )


In [19]:
env = dotenv_values(project_root / ".env")
try:
    host = "localhost"
    port = int(env.get("POSTGRES_PORT", 5432))# type:ignore
    database = env["POSTGRES_DB"]
    user = env["POSTGRES_USER"]
    password = env["POSTGRES_PASSWORD"]
    schema_name = "public"
    table_name = "pubseg"
except:
    raise

In [20]:
engine = build_postgres_engine(host, port, database, user, password)
metadata = MetaData(schema=schema_name)
pubseg = Table(table_name, metadata, autoload_with=engine, schema=schema_name)

### Tabela por quantidade de ocorrências ao ano

In [21]:
ano = func.extract("year", pubseg.c.data_referencia).cast(Integer)
evento = func.trim(pubseg.c.evento)

evento_counts_stmt = (
    select(
        evento.label("evento"),
        ano.label("ano"),
        func.count().label("quantidade"),
    )
    .where(pubseg.c.evento.is_not(None), evento != "")
    .group_by(evento, ano)
    .order_by(evento, ano)
)

evento_counts_por_ano = pd.read_sql_query(evento_counts_stmt, engine)
evento_counts_por_ano = (
    evento_counts_por_ano
    .pivot(index="evento", columns="ano", values="quantidade")
    .fillna(0)
    .astype(int)
    .reset_index()
)
evento_counts_por_ano.columns.name = None
evento_counts_por_ano

,evento,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Apreensão de Cocaína,636,648,648,648,648,648,648,648,648,648,648
1,Apreensão de Maconha,636,648,648,648,648,648,648,648,648,648,648
2,Arma de Fogo Apreendida,5724,5832,5832,5832,5832,5832,5832,5832,5832,5832,5832
3,Atendimento pré-hospitalar,324,324,324,324,324,324,324,324,324,324,324
4,Busca e salvamento,324,324,324,324,324,324,324,324,324,324,324
5,Combate a incêndios,324,324,324,324,324,324,324,324,324,324,324
6,Emissão de Alvarás de licença,324,324,324,324,324,324,324,324,324,324,324
7,Estupro,324,324,324,324,324,324,324,324,324,324,324
8,Estupro de vulnerável,324,324,324,324,324,324,324,324,324,324,324
9,Feminicídio,66600,67548,67548,67548,67560,67560,67560,67548,67548,67560,67558


plot feito pelo codex

In [22]:
eventos_para_plotar = [
    "Estupro",                                                              
    "Estupro de vulnerável",
    "Feminicídio",
    "Homicídio doloso",
    "Lesão corporal seguida de morte",
    "Tentativa de feminicídio",
    "Tentativa de homicídio",
]

eventos_por_ano_stmt = (
    select(
        ano.label("ano"),
        evento.label("evento"),
        func.count().label("quantidade"),
    )
    .where(evento.in_(eventos_para_plotar))
    .group_by(ano, evento)
    .order_by(ano, evento)
)

eventos_por_ano_df = pd.read_sql_query(eventos_por_ano_stmt, engine)

fig = px.bar(
    eventos_por_ano_df,
    x="ano",
    y="quantidade",
    color="evento",
    barmode="group",
    title="Ocorrências por ano para eventos selecionados",
    labels={"ano": "Ano", "quantidade": "Ocorrências", "evento": "Evento"},
)
fig.show()


In [23]:
anos_pubseg = list(range(2015, 2026))
pubseg_download_path = project_root / "data" / "xlsx_dowloads"

pubseg_frames = []
for ano in anos_pubseg:
    caminho_arquivo = pubseg_download_path / f"pubseg-{ano}.xlsx"
    db_ano = pd.read_excel(caminho_arquivo, usecols=["evento", "data_referencia"])
    db_ano["evento"] = db_ano["evento"].astype("string").str.strip()
    db_ano["ano"] = pd.to_datetime(db_ano["data_referencia"], errors="coerce").dt.year
    pubseg_frames.append(db_ano[["evento", "ano"]])

pubseg_xlsx_eventos_df = pd.concat(pubseg_frames, ignore_index=True)
pubseg_xlsx_eventos_df = pubseg_xlsx_eventos_df.dropna(subset=["evento", "ano"])
pubseg_xlsx_eventos_df = pubseg_xlsx_eventos_df.loc[pubseg_xlsx_eventos_df["evento"] != ""]

total_aparicoes_por_evento_ano_xlsx = (
    pubseg_xlsx_eventos_df
    .groupby(["evento", "ano"], as_index=False)
    .size()
    .rename(columns={"size": "total_aparicoes"})
)

total_aparicoes_por_evento_ano_xlsx_tabela = (
    total_aparicoes_por_evento_ano_xlsx
    .pivot(index="evento", columns="ano", values="total_aparicoes")
    .fillna(0)
    .astype(int)
    .reset_index()
)
total_aparicoes_por_evento_ano_xlsx_tabela.columns.name = None
total_aparicoes_por_evento_ano_xlsx_tabela

,evento,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Apreensão de Cocaína,636,648,648,648,648,648,648,648,648,648,648
1,Apreensão de Maconha,636,648,648,648,648,648,648,648,648,648,648
2,Arma de Fogo Apreendida,5724,5832,5832,5832,5832,5832,5832,5832,5832,5832,5832
3,Atendimento pré-hospitalar,324,324,324,324,324,324,324,324,324,324,324
4,Busca e salvamento,324,324,324,324,324,324,324,324,324,324,324
5,Combate a incêndios,324,324,324,324,324,324,324,324,324,324,324
6,Emissão de Alvarás de licença,324,324,324,324,324,324,324,324,324,324,324
7,Estupro,324,324,324,324,324,324,324,324,324,324,324
8,Estupro de vulnerável,324,324,324,324,324,324,324,324,324,324,324
9,Feminicídio,66600,67548,67548,67548,67560,67560,67560,67548,67548,67560,67558


In [ ]:
def download_xlsx(url: str, year: int, download_dir: str | Path) -> Path:
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    download_url = url.format(y=year)
    file_path = download_dir / f"pubseg-{year}.xlsx"
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    file_path.write_bytes(response.content)

    return file_path

In [ ]:
def table_ajust_pubseg(db: pd.DataFrame):

    year_series = pd.to_datetime(db["data_referencia"], errors="coerce").dt.year

    if year_series.isna().any():
        raise ValueError("The 'data_referencia' column must contain valid dates to generate the id_tabela.")
    
    row_id = pd.Series(range(1, len(db) + 1), index=db.index, dtype="int64")
    db["id_tabela"] = year_series.astype("Int64").astype("string") + row_id.astype("string")
    ordered_columns = ["id_tabela", *[column for column in db.columns if column != "id_tabela"]]
    db = db[ordered_columns]

    db["feminino"] = (
        pd.to_numeric(db["feminino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["masculino"] = (
        pd.to_numeric(db["masculino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["nao_informado"] = (
        pd.to_numeric(db["nao_informado"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["total_vitima"] = (
        pd.to_numeric(db["total_vitima"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    return db

In [ ]:
def tabel_ajust_disk(db):
    db.columns = (
        db.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    db["data_de_cadastro"] = pd.to_datetime(
        db["data_de_cadastro"], format="%d/%m/%Y %H:%M", errors="coerce"
    )
    db["sl_quantidade_vitimas"] = pd.to_numeric(
        db["sl_quantidade_vitimas"], errors="coerce"
    ).astype("Int64")
    db["id_tabela"] = db["data_de_cadastro"].dt.strftime("%Y%m%d%H%M%S")

In [ ]:
year = ["segundo-semestre-de-2025", "primeiro-semestre-de-2025"]
url = "https://www.gov.br/mdh/pt-br/acesso-a-informacao/dados-abertos/disque100/{y}"
download_path = project_root / "data" / "csv_dowloads"

schema_name = "public"
table_name = "raw_disk100"

path100_1=download_xlsx(url, year[0], download_path)
path100_2=download_xlsx(url, year[1], download_path)

# year = 2025
# url = "https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/download/dnsp-base-de-dados/bancovde-{y}.xlsx/@@download/file"
# download_path = project_root / "data" / "xlsx_dowloads"

# path_pubseg = download_xlsx(url, year, download_path)



In [ ]:
disk100_1 = pd.read_csv(path100_1, sep=";", encoding="latin1")
disk100_2 = pd.read_csv(path100_2, sep=";", encoding="latin1")
# pubseg = pd.read_excel(path_pubseg)

In [ ]:
disk100_1 = tabel_ajust_disk(disk100_1)
disk100_2 = tabel_ajust_disk(disk100_2)
# pubseg = table_ajust_pubseg(pubseg)

In [ ]:
from collections import Counter

anos_pubseg = list(range(2015, 2026))
pubseg_download_path = project_root / "data" / "xlsx_dowloads"

pubseg_bruto_por_ano = {
    ano: pd.read_excel(pubseg_download_path / f"pubseg-{ano}.xlsx")
    for ano in anos_pubseg
}

colunas_para_ignorar = {"id", "id_tabela"}
colunas_comparacao = sorted(
    set().union(*(set(df.columns) for df in pubseg_bruto_por_ano.values())) - colunas_para_ignorar
)

def normalizar_pubseg_para_comparacao(df: pd.DataFrame) -> pd.DataFrame:
    db = df.reindex(columns=colunas_comparacao).copy()

    if "data_referencia" in db.columns:
        data_referencia = pd.to_datetime(db["data_referencia"], errors="coerce")
        db["data_referencia"] = data_referencia.dt.strftime("%m-%d")

    for coluna in db.columns:
        serie = db[coluna]
        if pd.api.types.is_datetime64_any_dtype(serie):
            db[coluna] = pd.to_datetime(serie, errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
        elif pd.api.types.is_object_dtype(serie) or pd.api.types.is_string_dtype(serie):
            db[coluna] = serie.astype("string").str.strip()
        else:
            db[coluna] = serie.astype("string")

    return db.fillna("<NA>")

contadores_por_ano = {}
totais_por_ano = {}
for ano, db in pubseg_bruto_por_ano.items():
    db_normalizado = normalizar_pubseg_para_comparacao(db)
    linhas_normalizadas = list(db_normalizado.itertuples(index=False, name=None))
    contadores_por_ano[ano] = Counter(linhas_normalizadas)
    totais_por_ano[ano] = len(linhas_normalizadas)

matriz_sobreposicao_pubseg = pd.DataFrame(index=anos_pubseg, columns=anos_pubseg, dtype=float)

for ano_a in anos_pubseg:
    total_linhas_a = totais_por_ano[ano_a]
    contador_a = contadores_por_ano[ano_a]

    for ano_b in anos_pubseg:
        contador_b = contadores_por_ano[ano_b]
        linhas_iguais = sum(
            min(quantidade_a, contador_b.get(linha, 0))
            for linha, quantidade_a in contador_a.items()
        )
        percentual = 100 * linhas_iguais / total_linhas_a if total_linhas_a else 0
        matriz_sobreposicao_pubseg.loc[ano_a, ano_b] = percentual

matriz_sobreposicao_pubseg = matriz_sobreposicao_pubseg.round(2)

fig = px.imshow(
    matriz_sobreposicao_pubseg,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="Blues",
    labels={
        "x": "Ano B",
        "y": "Ano A",
        "color": "% de linhas de A presentes em B",
    },
    title="Sobreposição percentual entre arquivos pubseg-{ano}.xlsx",
)
fig.update_xaxes(side="top")
fig.show()

matriz_sobreposicao_pubseg
